# CRANE+ V2 ピック&プレース用 SmolVLA（Flow Matching）の学習

`reference/basic_final_assignment.ipynb` の 8章（モデル）・9章（学習ループ・action 正規化）をベースに、
学習データを自前の LeRobotDataset（`pixi run convert` で作ったもの）に差し替えたノートブック。

- Google Colab 無料枠の **T4 GPU** で動かす。学習は **最大 25 分** で打ち切る（30 分以内）。
- モデル定義はリポジトリの `model.py`（ノートブック 8章を切り出したもの）をアップロードして import する。
- 出力: `smolvla_ckpt.pt`（ダウンロードしてローカルの `checkpoints/` に置く）

### ★注意：作業が終わったら必ず「ランタイム」→「ランタイムを接続解除して削除」を実行すること

## 1. GPU を確認する
#### ランタイムのタイプを T4 GPU に変更してから実行してください。

In [ ]:
# ==============================================================
# GPU（T4）が使えることを確認する。無ければここで止める
# ==============================================================
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPUランタイムを選択してください "
        "(Colab メニュー: ランタイム → ランタイムのタイプを変更 → T4 GPU)"
    )
print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. LeRobot をインストールする
#### 変換（`pixi run convert`）と同じ LeRobot v0.6.0 を入れる。`[dataset]` で LeRobotDataset、`[smolvla]` で transformers（SmolVLM2 用）が入る。

In [ ]:
# ==============================================================
# 依存パッケージのインストール
# ==============================================================
# lerobot[dataset,smolvla]==0.6.0 … [dataset]: LeRobotDataset の読み込み、[smolvla]: transformers / num2words（SmolVLM2 の processor 用）
# torchao はリファレンスと同じく削除する（SmolVLM 周りと相性が悪いため）
!pip install -q "lerobot[dataset,smolvla]==0.6.0"
!pip uninstall -y -q torchao
# ※ pip が torch を入れ替えた場合は「ランタイム → セッションを再起動」してから 1 章から実行し直す

## 3. データセットと model.py をアップロードする
#### `crane_pp_dataset.zip`（`pixi run zip-dataset` で作成）と、リポジトリ直下の `model.py` の 2 つを選ぶ。

In [ ]:
# ==============================================================
# ローカルから zip と model.py をアップロードして展開する
# ==============================================================
import os
from pathlib import Path

from google.colab import files

# HF / transformers のログを静かにする（リファレンスと同じ）
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
# 長時間学習で起こりやすい断片化 OOM を回避（リファレンスと同じ）
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

WORKDIR = Path("/content")
# 前回アップロードした古いファイルを消す（同名ファイルがあると「model (1).py」の別名で保存され、
# import model が古い方を読み続けてしまうため）
for _f in list(WORKDIR.glob("model*.py")) + list(WORKDIR.glob("crane_pp_dataset*.zip")):
    _f.unlink()
uploaded = files.upload()  # ダイアログで crane_pp_dataset.zip と model.py を選ぶ
# 修正版の model.py か確認する（batch>1 対応の行があれば OK）
!grep -c "for img in pil_images" /content/model.py
!unzip -q -o /content/crane_pp_dataset.zip -d /content/
DATASET_ROOT = WORKDIR / "crane_pp"  # zip の中身（data/interim/crane_pp）
print(sorted(p.name for p in DATASET_ROOT.iterdir()))

## 4. SmolVLA を組み立てる
#### VLM（SmolVLM2-500M）は凍結し、Action head（Flow Matching）だけを学習する。構成はリファレンス 8.5 の教材用サイズと同じ。

In [ ]:
# ==============================================================
# 4. SmolVLA（model.py）を組み立てる
# ==============================================================
from model import SmolVLA, load_vlm_encoder

# T4 は bf16 非対応なので VLM は fp16 で読む（リファレンス 8.4 と同じ判定）
VLM_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
vlm_encoder = load_vlm_encoder("cuda", VLM_DTYPE)

smolvla = SmolVLA(
    vlm_encoder=vlm_encoder,
    action_dim=5,                 # CRANE+ V2: joint1〜4 + gripper
    state_dim=5,                  # /joint_states を同じ順に並べたもの
    chunk_size=10,                # 10 step = 1 秒 @10Hz（config/sim.yaml の chunk_size と同じ）
    num_transformer_layers=4,     # リファレンスの教材用サイズ
    num_heads=8,
    transformer_hidden_dim=256,
).to("cuda")

# 学習可能 params と凍結 params の内訳（frozen ≈ 500M、trainable ≈ 数 M のはず）
_train = sum(p.numel() for p in smolvla.parameters() if p.requires_grad)
_frozen = sum(p.numel() for p in smolvla.parameters() if not p.requires_grad)
print(f"SmolVLA — trainable: {_train/1e6:.2f}M  frozen: {_frozen/1e6:.2f}M")

## 5. データセットを読み込む
#### 各フレームに「そこから chunk_size 個先までの action」を付けて読む（リファレンス 9.1 と同じ）。

In [ ]:
# ==============================================================
# 5. 自前の LeRobotDataset（chunk_size ぶんの future action 付き）
# ==============================================================
from lerobot.datasets.lerobot_dataset import LeRobotDataset

CHUNK_SIZE_TRAIN = smolvla.chunk_size
DATASET_FPS = 10.0  # config/sim.yaml の fps と同じ
# i 番目の未来 action は i / fps 秒後（エピソード末尾を超える分は最後の値で埋められる）
delta_timestamps = {"action": [i / DATASET_FPS for i in range(CHUNK_SIZE_TRAIN)]}

# 動画は PyAV で読む。変換時の動画は AV1 形式で、Colab の torchcodec（既定）では
# 「Could not push packet to decoder: Invalid data found」となり読めなかったため
train_dataset = LeRobotDataset(
    "crane_pp", root=DATASET_ROOT, delta_timestamps=delta_timestamps, video_backend="pyav")

# 1 サンプルの中身を確認（action が (10, 5)、画像が (3, 256, 256) になっているか）
_sample = train_dataset[0]
for k, v in _sample.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:30s} shape={tuple(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:30s} val={str(v)[:60]}")
print(f"# episodes: {train_dataset.num_episodes}, # frames: {len(train_dataset)}")

## 6. action の mean / std で正規化する
#### リファレンス 9.2 は全サンプルを読み直して計算しているが、LeRobot が変換時に計算済みの統計量（`meta.stats`）を使えば動画を読み直さずに済むのでそれを使う。

In [ ]:
# ==============================================================
# 6. action の平均・標準偏差で正規化するヘルパ
# ==============================================================
action_mean = torch.tensor(train_dataset.meta.stats["action"]["mean"], dtype=torch.float32, device="cuda")
# std が 0 だと割り算で NaN が出るので下限を設ける（リファレンスと同じ）
action_std = torch.tensor(train_dataset.meta.stats["action"]["std"], dtype=torch.float32, device="cuda").clamp_min(1e-3)
print(f"action_mean: {action_mean.cpu().numpy().round(3)}")
print(f"action_std : {action_std.cpu().numpy().round(3)}")


def normalize_actions(a: torch.Tensor) -> torch.Tensor:
    # 学習前: (a - mean) / std で平均 0 / 分散 1 に近づける
    return (a - action_mean) / action_std

## 7. 1 step の時間を測って学習 step 数を決める
#### 学習を 25 分以内に収めるため、本番前に数 step だけ回して時間を測る。

In [ ]:
# ==============================================================
# 7. 1 step あたりの時間を計測し、25 分に収まる step 数を決める
# ==============================================================
import time

from torch.utils.data import DataLoader

TRAIN_BATCH_SIZE = 8      # VLM は推論のみ（勾配なし）なので T4 でも 8 は載る
TRAIN_NUM_WORKERS = 2     # 動画のデコードを並列にする（Colab は 2 CPU）。エラーが出たら 0 にする
TIME_BUDGET_SEC = 25 * 60 # 学習時間の上限（30 分以内に収める）

train_loader = DataLoader(
    train_dataset,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    num_workers=TRAIN_NUM_WORKERS,
    drop_last=True,
)


def compute_batch_loss(batch):
    """DataLoader の 1 バッチから Flow Matching loss を計算する（リファレンス 9.3 のループ本体）."""
    # 画像: LeRobotDataset は (B, C, H, W) float [0, 1] で返すので uint8 の (B, H, W, C) にする
    imgs = (batch["observation.images.top"] * 255).clamp(0, 255).to(torch.uint8)
    imgs = imgs.permute(0, 2, 3, 1).cpu().numpy()
    imgs_list = [imgs[i] for i in range(imgs.shape[0])]
    # タスク文・正規化した action chunk・現在の state
    tasks_list = list(batch["task"])
    actions_norm = normalize_actions(batch["action"].to("cuda", dtype=torch.float32))  # (B, K, 5)
    states = batch["observation.state"].to("cuda", dtype=torch.float32)  # (B, 5)
    return smolvla.compute_loss(imgs_list, tasks_list, states, actions_norm)


# 最初の数 step は CUDA の初期化で遅いので捨て、その後の 10 step で計測する（パラメタは更新しない）
smolvla.train()
_it = iter(train_loader)
_times = []
for i in range(13):
    _t0 = time.time()
    loss = compute_batch_loss(next(_it))
    loss.backward()
    smolvla.zero_grad()
    torch.cuda.synchronize()
    if i >= 3:
        _times.append(time.time() - _t0)
SEC_PER_STEP = sum(_times) / len(_times)
TRAIN_STEPS = int(TIME_BUDGET_SEC / SEC_PER_STEP)
print(f"{SEC_PER_STEP:.2f} s/step → TRAIN_STEPS = {TRAIN_STEPS}"
      f"（{TRAIN_STEPS * TRAIN_BATCH_SIZE / len(train_dataset):.1f} epoch 相当）")

## 8. 学習する
#### リファレンス 9.3 と同じ学習ループ（AdamW + warmup → cosine）。時間上限に達したら途中でも止める。

In [ ]:
# ==============================================================
# 8. PyTorch 学習ループ
# ==============================================================
import math

from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

TRAIN_LR = 3e-4          # peak lr（リファレンスと同じ）
TRAIN_WARMUP = 100       # linear warmup 区間
TRAIN_LOG_FREQ = 50      # loss を print する頻度

# optimizer: 学習可能な params（flow_head + vlm_projector + state_proj）だけを渡す
_trainable_params = [p for p in smolvla.parameters() if p.requires_grad]
optimizer = AdamW(_trainable_params, lr=TRAIN_LR, weight_decay=1e-4)


def _lr_lambda(step: int):
    # warmup 中は 0 → 1 に線形上昇、その後 cosine で 1.0 → 0.1
    if step < TRAIN_WARMUP:
        return step / max(TRAIN_WARMUP, 1)
    progress = min(max((step - TRAIN_WARMUP) / max(TRAIN_STEPS - TRAIN_WARMUP, 1), 0.0), 1.0)
    return 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * progress))


scheduler = LambdaLR(optimizer, lr_lambda=_lr_lambda)

smolvla.train()
_step = 0
_loss_history = []
_start = time.time()
print(f"Training {TRAIN_STEPS} steps (batch={TRAIN_BATCH_SIZE})...", flush=True)

# step 数か時間上限に達するまで、データセットを何周でも回す
while _step < TRAIN_STEPS and time.time() - _start < TIME_BUDGET_SEC:
    for batch in train_loader:
        if _step >= TRAIN_STEPS or time.time() - _start > TIME_BUDGET_SEC:
            break
        loss = compute_batch_loss(batch)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(_trainable_params, max_norm=1.0)  # 勾配爆発対策
        optimizer.step()
        scheduler.step()
        _loss_history.append(loss.item())

        if (_step + 1) % TRAIN_LOG_FREQ == 0 or _step == 0:
            _avg = sum(_loss_history[-TRAIN_LOG_FREQ:]) / min(len(_loss_history), TRAIN_LOG_FREQ)
            print(f"step {_step+1:5d}/{TRAIN_STEPS}  loss={_avg:.4f}  "
                  f"lr={scheduler.get_last_lr()[0]:.2e}  elapsed={time.time()-_start:.0f}s", flush=True)
        _step += 1

print(f"Training done: {_step} steps in {(time.time()-_start)/60:.1f} min")

## 9. loss を確認して checkpoint を保存・ダウンロードする

In [ ]:
# ==============================================================
# 9. loss curve の表示と checkpoint の保存
# ==============================================================
import matplotlib.pyplot as plt
import numpy as np

# loss が下がっているか目視で確認（移動平均で smoothing）
fig, ax = plt.subplots(figsize=(8, 4))
_y = np.array(_loss_history)
ax.plot(_y, alpha=0.4, linewidth=0.8, label="loss (raw)")
if len(_y) >= TRAIN_LOG_FREQ:
    ax.plot(np.arange(TRAIN_LOG_FREQ - 1, len(_y)),
            np.convolve(_y, np.ones(TRAIN_LOG_FREQ) / TRAIN_LOG_FREQ, mode="valid"),
            linewidth=1.5, label=f"moving avg (window={TRAIN_LOG_FREQ})")
ax.set_xlabel("step")
ax.set_ylabel("Flow Matching loss (MSE)")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

# 学習した部分 + action の統計量 + 構成を保存（リファレンス 9.3 と同じ中身。infer.py がこれを読む）
_ckpt = {
    "flow_head_state_dict": smolvla.flow_head.state_dict(),
    "vlm_projector_state_dict": smolvla.vlm_projector.state_dict(),
    "state_proj_state_dict": smolvla.state_proj.state_dict(),
    "action_mean": action_mean.cpu(),
    "action_std": action_std.cpu(),
    "chunk_size": smolvla.chunk_size,
    "action_dim": smolvla.action_dim,
    "state_dim": smolvla.state_dim,
    "hidden_dim": smolvla.hidden_dim,
    "loss_history": _loss_history,
}
torch.save(_ckpt, "/content/smolvla_ckpt.pt")
# ブラウザにダウンロードする → ローカルの checkpoints/smolvla_ckpt.pt に置く
files.download("/content/smolvla_ckpt.pt")